# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load and explore a FAIR^2 dataset describing mass spectrometry analysis of human extracellular vesicles using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and available at the following URL:

`https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json`

In [ ]:
# Ensure the required library is installed
!pip install -q mlcroissant

## 1. Data Loading
Load the dataset schema and metadata using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json

# The Croissant metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

# Display dataset title and description
print(f"Dataset title: {metadata.name}\n")
print(f"Dataset description: {metadata.description}")

### Metadata summary
* **Identifier**: 10.71728/senscience.vq4a-28xa
* **Version**: 1.0
* **License**: https://opendatacommons.org/licenses/by/1-0/
* **Published**: 2026-07-11


## 2. Data Overview
Review available record sets, their fields and IDs. All references in this notebook will be by `@id`, as required by the Croissant model.

In [ ]:
# Explore the list of available record sets
record_sets = list(metadata.record_sets)
print(f"Found {len(record_sets)} record set(s).\n")
for rset in record_sets:
    print(f"- Record set @id: {rset.id}")
    print(f"  Name: {rset.name}")
    print(f"  Description: {rset.description}\n")
    # List fields for first record set
    print("  Fields and their @id's:")
    for field in rset.fields:
        print(f"    - {field.id} (name: {field.name}, type: {field.data_type})")
    print()

**Tip:** The output above shows the available record sets and their fields, with their unique `@id` fields that are used for querying and data extraction.

## 3. Data Extraction
Load data from record sets using their `@id`. Data will be read into pandas DataFrames. We demonstrate on the primary record set (typically a table of proteins).

In [ ]:
# Extract all record sets into a dictionary of DataFrames
dataframes = {}
for rset in metadata.record_sets:
    print(f"Loading records for record set: {rset.id} ...")
    records = list(dataset.records(record_set=rset.id)) # The record_set parameter is always the @id
    df = pd.DataFrame(records)
    dataframes[rset.id] = df
    print(f"Shape: {df.shape}")
    print(f"Columns: {df.columns.tolist()}")
    print()
# If there's at least one record set, use the first for further analysis
main_record_set_id = metadata.record_sets[0].id if len(metadata.record_sets) > 0 else None
if main_record_set_id:
    print(f"Data preview for {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply filtering, normalization, and grouping operations. We select a numeric field by its `@id` (see field listing above, e.g., 'cr:normalizedAbundance' or 'cr:peptideCount') and a group field (e.g., 'cr:proteinDescription' or similar). _You may adjust field IDs to match those printed above._

In [ ]:
# Choose a numeric field and a grouping field
# Example: numeric_field_id = 'cr:normalizedAbundance'; group_field_id = 'cr:sampleType'
# Replace these with the actual @id's from your dataset fields

numeric_field_id = None
group_field_id = None
for field in metadata.record_sets[0].fields:
    # Use the first available numeric field and group field
    if numeric_field_id is None and field.data_type in ['Float', 'Number', 'Integer']:
        numeric_field_id = field.id
    if group_field_id is None and field.data_type in ['Text', 'String']:
        group_field_id = field.id

print(f"Using numeric field: {numeric_field_id}")
print(f"Using grouping field: {group_field_id}\n")
df = dataframes[main_record_set_id]
# Drop NA for simplicity
n_df = df.dropna(subset=[numeric_field_id])
# Filter records where the numeric field is above its median
threshold = n_df[numeric_field_id].median()
filtered_df = n_df[n_df[numeric_field_id] > threshold]

print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
display(filtered_df.head())

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (
    filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
) / filtered_df[numeric_field_id].std()
print(f"\nNormalized {numeric_field_id} for filtered records:")
display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Group by the selected text field (if not too many groups)
if group_field_id and group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    grouped_df = grouped_df.sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped (mean) {numeric_field_id} by {group_field_id}:")
    display(grouped_df.head(10))

## 5. Visualization
Let's visualize the distribution of the selected numeric field and the result of our grouping operation.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set(style="whitegrid")

if numeric_field_id in df.columns:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

if 'grouped_df' in locals() and not grouped_df.empty:
    plt.figure(figsize=(10, 5))
    top_groups = grouped_df.head(10)
    sns.barplot(y=group_field_id, x=numeric_field_id, data=top_groups)
    plt.title(f"Top 10 groups by average {numeric_field_id}")
    plt.xlabel(f"Mean {numeric_field_id}")
    plt.ylabel(group_field_id)
    plt.show()

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to:  
- Load a Croissant FAIR^2 dataset and examine its record sets and fields using their `@id`s.<br/>
- Extract data directly into pandas DataFrames for analysis.<br/>
- Conduct exploratory filtering, normalization, grouping, and plotting—all referencing data elements by their Croissant IDs.

This demonstrates an interoperable, robust workflow for sharing and exploring scientific datasets with Croissant and Python. You can extend this notebook for downstream statistical analysis or machine learning as appropriate.